# 02 — Bike Sharing Data Cleaning and Preparation

This notebook cleans and prepares the daily and hourly bike-sharing datasets for exploratory analysis and modeling.

The original raw datasets remain unchanged. All cleaning and transformation decisions are applied to separate working copies.

## 1. Load Raw Data and Define Project Paths

This section imports the required libraries, defines the project paths, verifies the source files, and loads the original datasets.

In [ ]:
from pathlib import Path
import pandas as pd

In [2]:
notebook_dir = Path.cwd()
project_root = notebook_dir.parents[1]
raw_data_dir = project_root / '01_data' / 'raw'
clean_data_dir = project_root / '01_data' / 'clean'
day_file = raw_data_dir / 'day.csv'
hour_file = raw_data_dir / 'hour.csv'


NameError: name 'Path' is not defined

In [ ]:
clean_data_dir.mkdir(
    parents=True,
    exist_ok=True)
print('clean data folder exists:', clean_data_dir.exists())

clean data folder exists: True


In [ ]:
print('day file exists:', day_file.exists())
print('hour file exists:', hour_file.exists())

day file exists: True
hour file exists: True


In [ ]:
source_file_check = pd.DataFrame({
    'file_name': [
        'day.csv',
        'hour.csv'
    ],

    'file_path': [
        str(day_file),
        str(hour_file)
    ],

    'file_exists': [
        day_file.exists(),
        hour_file.exists()
    ]
})

display(source_file_check)

,file_name,file_path,file_exists
0,day.csv,d:\کلاس دیتا\پاسخ تمارین\مصور سازی\fatemehkamr...,True
1,hour.csv,d:\کلاس دیتا\پاسخ تمارین\مصور سازی\fatemehkamr...,True


In [ ]:
day_raw = pd.read_csv(day_file)
hour_raw = pd.read_csv(hour_file)

In [ ]:
print('day raw shape:', day_raw.shape)
print('hour raw shape:', hour_raw.shape)

day raw shape: (731, 16)
hour raw shape: (17379, 17)


In [ ]:
display(day_raw.head())
display(hour_raw.head())

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


### Raw data loading result
Both source files were found and loaded successfully.
The daily dataset contains 731 rows and 16 columns, while the hourly dataset contains 17,379 rows and 17 columns. The observed dimensions match the results of the raw data audit.

## 2. Create Working Copies

This section creates independent copies of the raw datasets for cleaning and transformation.

In [ ]:
day_clean = day_raw.copy()
hour_clean = hour_raw.copy()
copy_summary = pd.DataFrame({
    'dataset': [
        'day',
        'hour'
    ],

    'raw_shape': [
        day_raw.shape,
        hour_raw.shape
    ],

    'clean_shape': [
        day_clean.shape,
        hour_clean.shape
    ],

    'independent_copy': [
        day_clean is not day_raw,
        hour_clean is not hour_raw
    ]
})

display(copy_summary)

,dataset,raw_shape,clean_shape,independent_copy
0,day,"(731, 16)","(731, 16)",True
1,hour,"(17379, 17)","(17379, 17)",True


### Working copy creation result
Independent working copies were created successfully. No rows, columns, or values were changed at this stage.

## 3. Convert Date Column to Datetime
This section converts the `dteday` column from text to a proper datetime data type in both working datasets.

In [ ]:
day_clean['dteday'] = pd.to_datetime(
    day_clean['dteday'],
    format='%Y-%m-%d',
    errors='raise'
)

hour_clean['dteday'] = pd.to_datetime(
    hour_clean['dteday'],
    format='%Y-%m-%d',
    errors='raise'
)


date_conversion_summary = pd.DataFrame({
    'dataset': [
        'day',
        'hour'
    ],

    'date_dtype': [
        str(day_clean['dteday'].dtype),
        str(hour_clean['dteday'].dtype)
    ],

    'minimum_date': [
        day_clean['dteday'].min(),
        hour_clean['dteday'].min()
    ],

    'maximum_date': [
        day_clean['dteday'].max(),
        hour_clean['dteday'].max()
    ]
})

display(date_conversion_summary)

,dataset,date_dtype,minimum_date,maximum_date
0,day,datetime64[us],2011-01-01,2012-12-31
1,hour,datetime64[us],2011-01-01,2012-12-31


### Date conversion result
The `dteday` column was successfully converted to datetime format in both working datasets.
The date range remains unchanged, covering January 1, 2011, through December 31, 2012.

## 4. Create Hourly Timestamp

This section combines the `dteday` and `hr` columns to create a complete hourly timestamp.

In [ ]:
hour_clean['datetime'] = (
    hour_clean['dteday']
    + pd.to_timedelta(
        hour_clean['hr'],
        unit='h'
    )
)

In [ ]:
hourly_datetime_summary = pd.DataFrame({
    'datetime_dtype': [
        str(hour_clean['datetime'].dtype)
    ],

    'minimum_datetime': [
        hour_clean['datetime'].min()
    ],

    'maximum_datetime': [
        hour_clean['datetime'].max()
    ],

    'missing_datetime': [
        hour_clean['datetime'].isna().sum()
    ]
})

display(hourly_datetime_summary)

,datetime_dtype,minimum_datetime,maximum_datetime,missing_datetime
0,datetime64[us],2011-01-01,2012-12-31 23:00:00,0


### Hourly timestamp creation result
A complete hourly timestamp was created successfully by combining the date and hour variables.
No invalid or missing timestamp values were produced.

## 5. Add Data Quality Flags

This section adds indicator columns for the data-quality issues identified during the audit. Original observations are retained without imputation or deletion.

In [ ]:
day_clean['zero_humidity_flag'] = (
    day_clean['hum']
    .eq(0)
    .astype('int8')
)

hour_clean['zero_humidity_flag'] = (
    hour_clean['hum']
    .eq(0)
    .astype('int8')
)

In [ ]:
hourly_records_per_day = (
    hour_clean.groupby('dteday')['hr']
    .transform('count')
)

hour_clean['incomplete_day_flag'] = (
    hourly_records_per_day < 24
).astype('int8')

In [ ]:
data_quality_flag_summary = pd.DataFrame({
    'flag': [
        'day zero humidity',
        'hour zero humidity',
        'hour records on incomplete days'
    ],

    'flagged_records': [
        day_clean['zero_humidity_flag'].sum(),
        hour_clean['zero_humidity_flag'].sum(),
        hour_clean['incomplete_day_flag'].sum()
    ]
})

display(data_quality_flag_summary)

incomplete_date_count = (
    hour_clean.loc[
        hour_clean['incomplete_day_flag'] == 1,
        'dteday'
    ]
    .nunique()
)

print('incomplete dates:', incomplete_date_count)

,flag,flagged_records
0,day zero humidity,1
1,hour zero humidity,22
2,hour records on incomplete days,1659


incomplete dates: 76


### Data quality flag result

Data-quality flags were added successfully.
The original zero-humidity observations and incomplete hourly dates were retained. The new indicator columns allow these records to be identified during exploratory analysis and modeling.

## 6. Create Interpretable Weather Variables

This section converts the normalized weather variables into more interpretable scales while retaining the original columns.

In [ ]:
for dataset in [day_clean, hour_clean]:
    dataset['temp_c'] = (
        47 * dataset['temp'] - 8
    )

    dataset['atemp_c'] = (
        66 * dataset['atemp'] - 16
    )

    dataset['humidity_pct'] = (
        100 * dataset['hum']
    )

    dataset['windspeed_original_scale'] = (
        67 * dataset['windspeed']
    )

In [ ]:
weather_transformation_summary = pd.DataFrame({
    'dataset': [
        'day',
        'hour'
    ],

    'temp_c_min': [
        day_clean['temp_c'].min(),
        hour_clean['temp_c'].min()
    ],

    'temp_c_max': [
        day_clean['temp_c'].max(),
        hour_clean['temp_c'].max()
    ],

    'humidity_pct_min': [
        day_clean['humidity_pct'].min(),
        hour_clean['humidity_pct'].min()
    ],

    'humidity_pct_max': [
        day_clean['humidity_pct'].max(),
        hour_clean['humidity_pct'].max()
    ]
})

display(weather_transformation_summary)

,dataset,temp_c_min,temp_c_max,humidity_pct_min,humidity_pct_max
0,day,-5.220871,32.498349,0.0,97.25
1,hour,-7.060000,39.000000,0.0,100.00


### Weather transformation result

Interpretable temperature, apparent temperature, humidity, and wind-speed variables were created successfully.

The original normalized weather variables were retained for reproducibility and possible modeling use.

## 7. Add Readable Category Labels

This section adds descriptive labels for the coded season, weekday, weather, and year variables while retaining the original codes.

In [ ]:
season_map = {
    1: 'spring',
    2: 'summer',
    3: 'fall',
    4: 'winter'
}

weekday_map = {
    0: 'sunday',
    1: 'monday',
    2: 'tuesday',
    3: 'wednesday',
    4: 'thursday',
    5: 'friday',
    6: 'saturday'
}

weather_map = {
    1: 'clear_or_partly_cloudy',
    2: 'mist_or_cloudy',
    3: 'light_rain_or_snow',
    4: 'heavy_rain_snow_or_fog'
}

year_map = {
    0: 2011,
    1: 2012
}

In [ ]:
for dataset in [day_clean, hour_clean]:
    dataset['year'] = dataset['yr'].map(year_map)
    dataset['season_label'] = dataset['season'].map(season_map)
    dataset['weekday_label'] = dataset['weekday'].map(weekday_map)
    dataset['weather_label'] = dataset['weathersit'].map(weather_map)

In [ ]:
category_label_validation = pd.DataFrame({
    'dataset': [
        'day',
        'hour'
    ],

    'missing_year_labels': [
        day_clean['year'].isna().sum(),
        hour_clean['year'].isna().sum()
    ],

    'missing_season_labels': [
        day_clean['season_label'].isna().sum(),
        hour_clean['season_label'].isna().sum()
    ],

    'missing_weekday_labels': [
        day_clean['weekday_label'].isna().sum(),
        hour_clean['weekday_label'].isna().sum()
    ],

    'missing_weather_labels': [
        day_clean['weather_label'].isna().sum(),
        hour_clean['weather_label'].isna().sum()
    ]
})

display(category_label_validation)

,dataset,missing_year_labels,missing_season_labels,missing_weekday_labels,missing_weather_labels
0,day,0,0,0,0
1,hour,0,0,0,0


### Category labeling result

Readable labels were added for the coded year, season, weekday, and weather variables.

All original codes were retained, and no unmapped category values were identified.

## 8. Sort Records Chronologically

This section sorts the daily and hourly datasets in chronological order and resets their row indexes.

In [ ]:
day_clean = (
    day_clean
    .sort_values('dteday')
    .reset_index(drop=True)
)

hour_clean = (
    hour_clean
    .sort_values('datetime')
    .reset_index(drop=True)
)

In [ ]:
sorting_validation = pd.DataFrame({
    'dataset': [
        'day',
        'hour'
    ],

    'chronologically_sorted': [
        day_clean['dteday'].is_monotonic_increasing,
        hour_clean['datetime'].is_monotonic_increasing
    ],

    'row_count': [
        len(day_clean),
        len(hour_clean)
    ]
})

display(sorting_validation)

,dataset,chronologically_sorted,row_count
0,day,True,731
1,hour,True,17379


### Chronological sorting result

Both working datasets were sorted successfully in chronological order.

The original number of observations was preserved.

## 9. Final Cleaning Validation

This section verifies that the cleaning and transformation steps were completed successfully without changing the original rental observations.

In [ ]:
day_created_columns = [
    'zero_humidity_flag',
    'temp_c',
    'atemp_c',
    'humidity_pct',
    'windspeed_original_scale',
    'year',
    'season_label',
    'weekday_label',
    'weather_label'
]

hour_created_columns = [
    'datetime',
    'zero_humidity_flag',
    'incomplete_day_flag',
    'temp_c',
    'atemp_c',
    'humidity_pct',
    'windspeed_original_scale',
    'year',
    'season_label',
    'weekday_label',
    'weather_label'
]

cleaning_validation = pd.DataFrame({
    'check': [
        'day row count preserved',
        'hour row count preserved',
        'day rental totals preserved',
        'hour rental totals preserved',
        'day date type is datetime',
        'hour date and timestamp types are datetime',
        'day created columns contain no missing values',
        'hour created columns contain no missing values',
        'zero-humidity flags match audit results',
        'incomplete hourly dates match audit results'
    ],

    'passed': [
        len(day_clean) == len(day_raw),

        len(hour_clean) == len(hour_raw),

        day_clean[['casual', 'registered', 'cnt']]
        .sum()
        .equals(
            day_raw[['casual', 'registered', 'cnt']].sum()
        ),

        hour_clean[['casual', 'registered', 'cnt']]
        .sum()
        .equals(
            hour_raw[['casual', 'registered', 'cnt']].sum()
        ),

        pd.api.types.is_datetime64_any_dtype(
            day_clean['dteday']
        ),

        (
            pd.api.types.is_datetime64_any_dtype(
                hour_clean['dteday']
            )
            and
            pd.api.types.is_datetime64_any_dtype(
                hour_clean['datetime']
            )
        ),

        day_clean[
            day_created_columns
        ].isna().sum().sum() == 0,

        hour_clean[
            hour_created_columns
        ].isna().sum().sum() == 0,

        (
            day_clean['zero_humidity_flag'].sum() == 1
            and
            hour_clean['zero_humidity_flag'].sum() == 22
        ),

        (
            hour_clean.loc[
                hour_clean['incomplete_day_flag'] == 1,
                'dteday'
            ].nunique() == 76
        )
    ]
})

display(cleaning_validation)

,check,passed
0,day row count preserved,True
1,hour row count preserved,True
2,day rental totals preserved,True
3,hour rental totals preserved,True
4,day date type is datetime,True
5,hour date and timestamp types are datetime,True
6,day created columns contain no missing values,True
7,hour created columns contain no missing values,True
8,zero-humidity flags match audit results,True
9,incomplete hourly dates match audit results,True


In [ ]:
print(
    'all cleaning checks passed:',
    cleaning_validation['passed'].all()
)

all cleaning checks passed: True


### Final cleaning validation result

All cleaning validation checks passed successfully.

The original number of observations and rental totals were preserved. The new date, timestamp, weather, category-label, and data-quality flag variables were created without introducing missing values.

## 10. Export Clean Datasets

This section exports the prepared daily and hourly datasets to the clean data folder.

In [ ]:
day_clean_file = clean_data_dir / 'day_clean.csv'
hour_clean_file = clean_data_dir / 'hour_clean.csv'

In [ ]:
day_clean.to_csv(
    day_clean_file,
    index=False
)

hour_clean.to_csv(
    hour_clean_file,
    index=False
)

print('clean datasets were exported successfully.')

clean datasets were exported successfully.


In [ ]:
export_summary = pd.DataFrame({
    'file_name': [
        day_clean_file.name,
        hour_clean_file.name
    ],

    'file_exists': [
        day_clean_file.exists(),
        hour_clean_file.exists()
    ],

    'row_count': [
        len(day_clean),
        len(hour_clean)
    ],

    'column_count': [
        day_clean.shape[1],
        hour_clean.shape[1]
    ]
})

display(export_summary)

,file_name,file_exists,row_count,column_count
0,day_clean.csv,True,731,25
1,hour_clean.csv,True,17379,28


### Data cleaning completion

The daily and hourly datasets were prepared and exported successfully.

No original rental records were deleted or imputed. The raw source files remained unchanged, and all documented data-quality issues were retained with explicit indicator variables.

The clean datasets are ready for exploratory data analysis.